# 02 · Features

We turn the clean dataset from notebook 01 into model-ready features, using only information available **before or during a film's release week**.

| Group | Features | Source |
|---|---|---|
| Structured | budget, runtime, release date, genres, language, country, counts, sequel flag | TMDB fields |
| Track record | how experienced each film's cast and key crew are, and how well their **earlier** films did | cast and crew lists |
| Plot | sentence embedding of the plot summary | `overview` |
| Keywords | sentence embedding of the keyword list | `Keywords` |

The last three groups process the cast, crew and text fields; notebook 03 compares models with and without them.

**Output:** `features.parquet` (one row per film, keyed on `imdb_id`, with the target and the split).

1. Imports
2. Load
3. Structured features
4. Sequels
5. Cast and crew track record
6. Text embeddings: plot and keywords
7. Assemble and save

## 1. Imports

In [1]:
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
sns.set_theme(style="whitegrid", context="notebook")

## 2. Load

`split` comes from notebook 01. Anything we *learn* from the data in this notebook (most common categories, the dimension reduction of the embeddings) is fitted on the **train** films only, so the validation and test films stay unseen.

In [2]:
df = pd.read_parquet("../data/processed/movies_clean.parquet")
is_train = df["split"] == "train"
print(df.shape)
df["split"].value_counts()

(6090, 26)


split
train    4263
test      914
val       913
Name: count, dtype: int64

In [3]:
df.head()

,imdb_id,title,orig_split,release_date,year,runtime,original_language,genres,production_companies,production_countries,spoken_languages,belongs_to_collection,Keywords,overview,cast,crew,budget,budget_cpi,log_budget_cpi,revenue,revenue_source,revenue_tmdb_2019,revenue_cpi,log_revenue_cpi,roi,split
0,tt2637294,Hot Tub Time Machine 2,train,2015-02-20,2015.0,93.0,en,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Paramount Pictures', 'id': 4}, {'name': 'Unit...","[{'iso_3166_1': 'US', 'name': 'United States of America'}]","[{'iso_639_1': 'en', 'name': 'English'}]","[{'id': 313576, 'name': 'Hot Tub Time Machine Collection...","[{'id': 4379, 'name': 'time travel'}, {'id': 9663, 'name...","When Lou, who has become the ""father of the Internet,"" i...","[{'cast_id': 4, 'character': 'Lou', 'credit_id': '52fe4e...","[{'credit_id': '59ac067c92514107af02c8c8', 'department':...",14000000.0,1.901637e+07,16.760811,13081651.0,imdb,12314651.0,1.776896e+07,16.692964,0.934404,train
1,tt0368933,The Princess Diaries 2: Royal Engagement,train,2004-08-06,2004.0,113.0,en,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama...","[{'name': 'Walt Disney Pictures', 'id': 2}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]","[{'iso_639_1': 'en', 'name': 'English'}]","[{'id': 107674, 'name': 'The Princess Diaries Collection...","[{'id': 2505, 'name': 'coronation'}, {'id': 4263, 'name'...",Mia Thermopolis is now a college graduate and on her way...,"[{'cast_id': 1, 'character': 'Mia Thermopolis', 'credit_...","[{'credit_id': '52fe43fe9251416c7502563d', 'department':...",40000000.0,6.817215e+07,18.037547,134734481.0,imdb,95149435.0,2.296285e+08,19.251973,3.368362,train
2,tt2582802,Whiplash,train,2014-10-10,2014.0,105.0,en,"[{'id': 18, 'name': 'Drama'}]","[{'name': 'Bold Films', 'id': 2266}, {'name': 'Blumhouse...","[{'iso_3166_1': 'US', 'name': 'United States of America'}]","[{'iso_639_1': 'en', 'name': 'English'}]",NaN,"[{'id': 1416, 'name': 'jazz'}, {'id': 1523, 'name': 'obs...","Under the direction of a ruthless instructor, a talented...","[{'cast_id': 5, 'character': 'Andrew Neimann', 'credit_i...","[{'credit_id': '54d5356ec3a3683ba0000039', 'department':...",3300000.0,4.487750e+06,15.316862,50360880.0,imdb,13092000.0,6.848698e+07,18.042154,15.260873,train
3,tt0093743,Pinocchio and the Emperor of the Night,train,1987-08-06,1987.0,83.0,en,"[{'id': 16, 'name': 'Animation'}, {'id': 12, 'name': 'Ad...",NaN,NaN,"[{'iso_639_1': 'en', 'name': 'English'}]",NaN,NaN,"Pinocchio and his friends, a glow worm and a marionette,...","[{'cast_id': 6, 'character': 'Pinocchio (voice)', 'credi...","[{'credit_id': '52fe46f49251416c9106558b', 'department':...",8000000.0,2.267204e+07,16.936643,3261638.0,imdb,3261638.0,9.243499e+06,16.039431,0.407705,train
4,tt0431021,The Possession,train,2012-08-30,2012.0,92.0,en,"[{'id': 27, 'name': 'Horror'}, {'id': 53, 'name': 'Thril...","[{'name': 'Ghost House Pictures', 'id': 768}, {'name': '...","[{'iso_3166_1': 'US', 'name': 'United States of America'...","[{'iso_639_1': 'en', 'name': 'English'}]",NaN,NaN,"A young girl buys an antique box at a yard sale, unaware...","[{'cast_id': 23, 'character': 'Clyde', 'credit_id': '52f...","[{'credit_id': '52fe4981c3a368484e12ee29', 'department':...",14000000.0,1.963118e+07,16.792630,85446075.0,imdb,85446075.0,1.198148e+08,18.601458,6.103291,train


In [4]:
def parse(s):
    """TMDB list columns are stored as text such as "[{'name': 'Comedy'}]"."""
    if not isinstance(s, str):
        return []
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []


for col in ["genres", "production_companies", "production_countries", "spoken_languages",
            "belongs_to_collection", "Keywords", "cast", "crew"]:
    df[col] = df[col].map(parse)

## 3. Structured features

Missing values (e.g. an unknown budget, 26% of films) stay as `NaN`.

In [5]:
F = pd.DataFrame(index=df.index)

F["log_budget_cpi"] = df["log_budget_cpi"]
F["runtime"] = df["runtime"].replace(0, np.nan)
F["year"] = df["year"]
F["month"] = df["release_date"].dt.month
F["weekday"] = df["release_date"].dt.dayofweek
F["n_genres"] = df["genres"].str.len()
F["n_production_companies"] = df["production_companies"].str.len()
F["n_countries"] = df["production_countries"].str.len()
F["n_languages"] = df["spoken_languages"].str.len()

F.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
log_budget_cpi,4489.0,17.29,1.35,10.06,16.61,17.49,18.23,20.11
runtime,6088.0,107.43,18.59,46.00,95.00,104.00,117.00,254.00
year,6090.0,2002.79,9.91,1980.00,1995.00,2005.00,2011.00,2017.00
month,6090.0,6.84,3.35,1.00,4.00,7.00,10.00,12.00
weekday,6090.0,3.29,1.29,0.00,2.00,4.00,4.00,6.00
n_genres,6090.0,2.52,1.12,0.00,2.00,2.00,3.00,8.00
n_production_companies,6090.0,2.95,2.26,0.00,1.00,2.00,4.00,26.00
n_countries,6090.0,1.37,0.83,0.00,1.00,1.00,2.00,12.00
n_languages,6090.0,1.46,0.90,0.00,1.00,1.00,2.00,9.00


For original language and main production country, the 8 most common values among the training films each get a 0/1 column; all other values are grouped as "other". Each genre gets its own 0/1 column, since a film can have several.

In [6]:
def one_hot_top(values, prefix, k=8):
    top = values[is_train].value_counts().index[:k]
    return pd.get_dummies(values.where(values.isin(top), "other"), prefix=prefix, dtype=int)


country_main = df["production_countries"].map(lambda L: L[0]["iso_3166_1"] if L else "none")
cat = pd.concat([one_hot_top(df["original_language"], "lang"),
                 one_hot_top(country_main, "country")], axis=1)

genre_names = df["genres"].map(lambda L: [g["name"] for g in L])
genre = (pd.get_dummies(genre_names.explode()).groupby(level=0).max()
           .reindex(df.index, fill_value=0).astype(int))
genre.columns = ["genre_" + c.replace(" ", "_") for c in genre.columns]

print(f"language / country columns: {cat.shape[1]}   genre columns: {genre.shape[1]}")

language / country columns: 18   genre columns: 19


In [7]:
cat.head()

,lang_cn,lang_en,lang_es,lang_fr,lang_hi,lang_ja,lang_ko,lang_other,lang_zh,country_AU,country_CA,country_DE,country_FR,country_GB,country_IN,country_US,country_none,country_other
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [8]:
genre.head()

,genre_Action,genre_Adventure,genre_Animation,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Family,genre_Fantasy,genre_Foreign,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science_Fiction,genre_Thriller,genre_War,genre_Western
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0


## 4. Sequels

TMDB marks every film in a franchise as part of a collection, including the first, which only joined once a sequel was made, i.e. after it succeeded. We therefore mark a film as a sequel only if another film of the same collection was released **before** it. When earlier films are not in our data, we fall back on the title (a number or "Part", e.g. *Hot Tub Time Machine 2*).

In [9]:
collection = df["belongs_to_collection"].map(lambda L: L[0]["name"] if L else None)
first_date = df.groupby(collection)["release_date"].transform("min")
later_in_collection = collection.notna() & (df["release_date"] > first_date)
sequel_title = df["title"].str.contains(r"\b(?:[2-9]|II|III|IV|V|VI|Part)\b", regex=True, na=False)

F["is_sequel"] = (later_in_collection | (collection.notna() & sequel_title)).astype(int)
pd.crosstab(collection.notna().rename("in a collection"), F["is_sequel"])

is_sequel,0,1
in a collection,,
False,4854,0
True,602,634


## 5. Cast and crew track record

For each film and each role we describe the people involved by their record **before** this film's release:

- `exp_<role>`: how many of them had at least 2 earlier films in our data (experience).
- `prev_rev_<role>`: the average ln revenue (2025 dollars) of their earlier films, over people with at least one earlier film. `NaN` if nobody has one.

Only films released strictly before the film count, so no feature uses information from the future. Revenue history comes from train and validation films only; **test revenues are never used**.

| Role | Who |
|---|---|
| actor | the 10 top-billed cast members |
| director | job "Director" |
| producer | jobs "Producer" and "Executive Producer" |
| writer | Writing department |
| composer | job "Original Music Composer" |
| cinematographer | job "Director of Photography" |
| editor | job "Editor" |

In [10]:
role_people = {
    "actor": df["cast"].map(lambda L: [p["name"] for p in L if p["order"] < 10]),
    "director": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Director"]),
    "producer": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] in ("Producer", "Executive Producer")]),
    "writer": df["crew"].map(lambda L: [p["name"] for p in L if p["department"] == "Writing"]),
    "composer": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Original Music Composer"]),
    "cinematographer": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Director of Photography"]),
    "editor": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Editor"]),
}

MIN_PRIOR = 2
known_revenue = df["log_revenue_cpi"].where(df["split"] != "test")   # test revenues hidden


def track_record(people):
    """Per film: number of experienced people, and mean ln revenue of their earlier films."""
    long = (people.map(lambda L: list(dict.fromkeys(L)))          # each person once per film
                  .explode().dropna().rename("person").to_frame())
    long["film"] = long.index
    long["date"] = df.loc[long.index, "release_date"].values
    long["rev"] = known_revenue.loc[long.index].values

    # Totals per person and release date, then running totals over strictly earlier dates
    per_date = (long.groupby(["person", "date"])
                    .agg(n=("film", "size"), rev_sum=("rev", "sum"), rev_cnt=("rev", "count"))
                    .reset_index())
    g = per_date.groupby("person")
    per_date["n_prior"] = g["n"].cumsum() - per_date["n"]
    per_date["rev_sum_prior"] = g["rev_sum"].cumsum() - per_date["rev_sum"]
    per_date["rev_cnt_prior"] = g["rev_cnt"].cumsum() - per_date["rev_cnt"]

    long = long.merge(per_date[["person", "date", "n_prior", "rev_sum_prior", "rev_cnt_prior"]],
                      on=["person", "date"], how="left")
    long["prev_rev"] = (long["rev_sum_prior"] / long["rev_cnt_prior"]).where(long["rev_cnt_prior"] > 0)

    by_film = long.groupby("film")
    out = pd.DataFrame({
        "exp": by_film["n_prior"].apply(lambda s: (s >= MIN_PRIOR).sum()),
        "prev_rev": by_film["prev_rev"].mean(),
    })
    return out.reindex(df.index).fillna({"exp": 0})


track = pd.DataFrame(index=df.index)
for role, people in role_people.items():
    t = track_record(people)
    track[f"exp_{role}"] = t["exp"].astype(int)
    track[f"prev_rev_{role}"] = t["prev_rev"]

track.describe().T[["count", "mean", "50%", "max"]].round(2)

,count,mean,50%,max
exp_actor,6090.0,4.35,4.00,11.00
prev_rev_actor,5667.0,17.67,17.77,21.76
exp_director,6090.0,0.39,0.00,6.00
prev_rev_director,3221.0,17.82,18.12,21.49
exp_producer,6090.0,1.57,1.00,17.00
prev_rev_producer,4040.0,17.96,18.21,21.49
exp_writer,6090.0,0.57,0.00,8.00
prev_rev_writer,3281.0,17.98,18.23,21.70
exp_composer,6090.0,0.39,0.00,3.00
prev_rev_composer,2639.0,18.12,18.37,21.49


A quick check that the revenue history carries signal: correlation of each `prev_rev_<role>` with the film's own ln revenue, on training films.

In [11]:
track.loc[is_train, [c for c in track if c.startswith("prev_rev")]] \
     .corrwith(df.loc[is_train, "log_revenue_cpi"]).round(2).rename("correlation with ln revenue")

prev_rev_actor              0.28
prev_rev_director           0.47
prev_rev_producer           0.53
prev_rev_writer             0.43
prev_rev_composer           0.37
prev_rev_cinematographer    0.38
prev_rev_editor             0.43
Name: correlation with ln revenue, dtype: float64

## 6. Text embeddings: plot and keywords

We encode each film's plot summary (`overview`) and its keyword list with a pretrained sentence-embedding model, **all-MiniLM-L6-v2** (Sentence-BERT). It maps a text to a 384-dimensional vector so that texts with similar meaning get similar vectors. The model is pretrained on general text and does not see our revenues.

A few TMDB keywords are added by users after release and describe reception rather than content; we remove them before encoding.

The model is downloaded on first use (about 90 MB).

In [12]:
POST_RELEASE_KEYWORDS = {"duringcreditsstinger", "aftercreditsstinger", "box office flop",
                         "cult film", "flop", "blockbuster", "oscar winner"}

overview_text = df["overview"].fillna("")
keyword_text = df["Keywords"].map(
    lambda L: ", ".join(k["name"] for k in L if k["name"].lower() not in POST_RELEASE_KEYWORDS))

print(f"films with no overview: {(overview_text == '').sum()}   with no keywords: {(keyword_text == '').sum()}")
keyword_text.head()

films with no overview: 4   with no keywords: 438


0                                   time travel, sequel, hot tub
1                    coronation, duty, marriage, falling in love
2    jazz, obsession, conservatory, music teacher, new york c...
3                                                               
4                                                               
Name: Keywords, dtype: str

In [13]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
TEXT_METHOD = "Sentence-BERT (all-MiniLM-L6-v2)"


def embed(texts):
    return encoder.encode(texts.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)


print("Text method:", TEXT_METHOD)
overview_vec = embed(overview_text)
keyword_vec = embed(keyword_text)
print(overview_vec.shape, keyword_vec.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Text method: Sentence-BERT (all-MiniLM-L6-v2)


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

(6090, 384) (6090, 384)


384 dimensions is a lot for about 4,000 training films, so we reduce each embedding to **32 dimensions** with principal component analysis (PCA), fitted on the training films. Films with no text get `NaN`.

In [14]:
N_COMPONENTS = 32


def reduce(vectors, empty, prefix):
    pca = PCA(n_components=N_COMPONENTS, random_state=42).fit(vectors[(is_train & ~empty).values])
    out = pd.DataFrame(pca.transform(vectors), index=df.index,
                       columns=[f"{prefix}_{i}" for i in range(N_COMPONENTS)])
    out[empty] = np.nan
    print(f"{prefix}: {pca.explained_variance_ratio_.sum():.0%} of the variance kept")
    return out


plot_emb = reduce(overview_vec, overview_text == "", "plot_emb")
kw_emb = reduce(keyword_vec, keyword_text == "", "kw_emb")

plot_emb: 42% of the variance kept
kw_emb: 55% of the variance kept


## 7. Assemble and save

One row per film: identifiers, split, target, then the feature groups. The column-name prefixes identify the groups in notebook 03.

In [15]:
features = pd.concat([
    df[["imdb_id", "title", "split", "log_revenue_cpi", "revenue_cpi"]],
    F, cat, genre, track, plot_emb, kw_emb,
], axis=1)

groups = {
    "structured": list(F.columns) + list(cat.columns) + list(genre.columns),
    "track record": list(track.columns),
    "plot embedding": list(plot_emb.columns),
    "keyword embedding": list(kw_emb.columns),
}
print(f"features.parquet: {features.shape}   text method: {TEXT_METHOD}")
pd.Series({g: len(c) for g, c in groups.items()}, name="columns")

features.parquet: (6090, 130)   text method: Sentence-BERT (all-MiniLM-L6-v2)


structured           47
track record         14
plot embedding       32
keyword embedding    32
Name: columns, dtype: int64

In [17]:
features.head()

,imdb_id,title,split,log_revenue_cpi,revenue_cpi,log_budget_cpi,runtime,year,month,weekday,n_genres,n_production_companies,n_countries,n_languages,is_sequel,lang_cn,lang_en,lang_es,lang_fr,lang_hi,lang_ja,lang_ko,lang_other,lang_zh,country_AU,country_CA,country_DE,country_FR,country_GB,country_IN,country_US,country_none,country_other,genre_Action,genre_Adventure,genre_Animation,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Family,genre_Fantasy,genre_Foreign,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science_Fiction,genre_Thriller,genre_War,genre_Western,exp_actor,prev_rev_actor,exp_director,prev_rev_director,exp_producer,prev_rev_producer,exp_writer,prev_rev_writer,exp_composer,prev_rev_composer,exp_cinematographer,prev_rev_cinematographer,exp_editor,prev_rev_editor,plot_emb_0,plot_emb_1,plot_emb_2,plot_emb_3,plot_emb_4,plot_emb_5,plot_emb_6,plot_emb_7,plot_emb_8,plot_emb_9,plot_emb_10,plot_emb_11,plot_emb_12,plot_emb_13,plot_emb_14,plot_emb_15,plot_emb_16,plot_emb_17,plot_emb_18,plot_emb_19,plot_emb_20,plot_emb_21,plot_emb_22,plot_emb_23,plot_emb_24,plot_emb_25,plot_emb_26,plot_emb_27,plot_emb_28,plot_emb_29,plot_emb_30,plot_emb_31,kw_emb_0,kw_emb_1,kw_emb_2,kw_emb_3,kw_emb_4,kw_emb_5,kw_emb_6,kw_emb_7,kw_emb_8,kw_emb_9,kw_emb_10,kw_emb_11,kw_emb_12,kw_emb_13,kw_emb_14,kw_emb_15,kw_emb_16,kw_emb_17,kw_emb_18,kw_emb_19,kw_emb_20,kw_emb_21,kw_emb_22,kw_emb_23,kw_emb_24,kw_emb_25,kw_emb_26,kw_emb_27,kw_emb_28,kw_emb_29,kw_emb_30,kw_emb_31
0,tt2637294,Hot Tub Time Machine 2,train,16.692964,1.776896e+07,16.760811,93.0,2015.0,2,4,1,3,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,17.570353,1,18.156794,3,16.340907,0,18.376133,1,18.667910,1,16.571461,0,NaN,-0.062508,-0.111420,-0.065433,0.116423,0.019369,-0.104030,0.121908,-0.128604,0.025463,-0.007399,-0.038566,0.064159,-0.071711,-0.018959,0.075913,0.040611,0.032884,0.097330,-0.007514,0.112095,-0.022648,-0.173328,-0.090450,0.030986,0.086306,0.112700,0.005668,0.059804,0.113757,0.025081,-0.031949,-0.062090,-0.195071,-0.185455,0.096846,-0.152812,-0.042502,0.006605,0.182836,-0.015272,-0.154397,-0.275297,0.033731,0.049432,0.034164,-0.134427,-0.061855,0.075031,0.005559,0.058693,-0.015133,0.042816,-0.113175,0.018185,0.093198,-0.005373,-0.127654,-0.117189,-0.109443,-0.020643,0.053172,0.024197,-0.160732,0.096973
1,tt0368933,The Princess Diaries 2: Royal Engagement,train,19.251973,2.296285e+08,18.037547,113.0,2004.0,8,4,4,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,7,18.404282,1,18.905393,2,18.999717,0,NaN,1,19.151518,1,16.401302,1,19.052723,0.191206,0.164139,0.023427,0.002327,0.156879,0.409230,-0.042349,-0.081771,-0.057501,0.048478,-0.124190,-0.138181,0.025185,-0.080669,0.116747,-0.055638,0.067799,-0.020937,-0.146953,-0.075402,0.198858,-0.102909,0.133979,-0.034959,-0.110484,-0.007999,0.057801,-0.094679,-0.116774,0.150045,-0.000913,0.041885,-0.074491,0.258352,0.045255,-0.216313,0.076449,0.087227,-0.077833,0.100877,-0.021852,0.077254,-0.342306,0.168877,-0.120907,0.030341,-0.025309,-0.012271,0.108294,0.079179,-0.105784,0.067236,0.092834,0.008837,-0.057392,0.025866,0.124082,0.059315,-0.097394,-0.027178,-0.101036,0.058693,-0.032899,0.008387
2,tt2582802,Whiplash,train,18.042154,6.848698e+07,15.316862,105.0,2014.0,10,4,1,3,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,4,17.624404,0,NaN,6,17.408760,0,17.359979,0,NaN,0,14.129203,0,NaN,-0.077642,0.040498,-0.153661,-0.039604,0.060001,-0.047333,-0.057259,0.044460,-0.324450,0.014540,0.231527,-0.023926,0.005047,-0.014180,-0.071118,0.035353,0.020519,-0.111579,0.014245,-0.079557,-0.031850,-0.061978,0.018281,0.008702,-0.121350,-0.011007,-0.052605,-0.027209,0.015422,-0.029724,0.008565,-0.060834,-0.020091,0.125121,-0.022938,0.022244,-0.264610,-0.040154,0.250465,-0.022047,0.119152,0.080023,0.282114,-0.064578,-0.186748,-0.003742,-0.047639,-0.049359,0.032722,-0.258588,-0.038523,-0.030843,0.149874,-0.252313,0.029879,

In [16]:
features.attrs["text_method"] = TEXT_METHOD
features.to_parquet("../data/processed/features.parquet", index=False)